In [1]:
import folium
import pandas as pd 
from scrapper import Scrapper
df = pd.read_csv("datos_inmuebles.csv")



In [2]:
df.isna().sum()

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1980 entries, 0 to 1979
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                1980 non-null   int64  
 1   url               1980 non-null   object 
 2   precio            1980 non-null   object 
 3   expensas          1750 non-null   object 
 4   area_m2           1844 non-null   object 
 5   dormitorios       1444 non-null   object 
 6   antiguedad        1448 non-null   object 
 7   puntaje_arg_prop  1980 non-null   int64  
 8   imagen_path       1973 non-null   object 
 9   image_url         1980 non-null   object 
 10  lat               1923 non-null   float64
 11  lon               1923 non-null   float64
dtypes: float64(2), int64(2), object(8)
memory usage: 185.8+ KB


In [3]:
df = df[ df["lat"].notna() & df["lon"].notna() ]
df.info()
df[['lat','lon']].value_counts().head(100)


<class 'pandas.core.frame.DataFrame'>
Index: 1923 entries, 0 to 1979
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                1923 non-null   int64  
 1   url               1923 non-null   object 
 2   precio            1923 non-null   object 
 3   expensas          1699 non-null   object 
 4   area_m2           1788 non-null   object 
 5   dormitorios       1408 non-null   object 
 6   antiguedad        1407 non-null   object 
 7   puntaje_arg_prop  1923 non-null   int64  
 8   imagen_path       1916 non-null   object 
 9   image_url         1923 non-null   object 
 10  lat               1923 non-null   float64
 11  lon               1923 non-null   float64
dtypes: float64(2), int64(2), object(8)
memory usage: 195.3+ KB


lat         lon       
-34.595300  -58.405130    4
-34.607490  -58.500170    4
-34.544895  -58.472244    4
-34.586180  -58.414790    4
-34.619410  -58.452000    4
                         ..
-34.553870  -58.455360    2
-34.581340  -58.494080    2
-34.584220  -58.438280    2
-34.642300  -58.503700    2
-34.642060  -58.438990    2
Name: count, Length: 100, dtype: int64

La columna "dormitorios" tiene valores nulos cuando son monoambientes y valores del tipo string "1 dorm. , 2 dorm, etc" cuando no es monoambinete. Por esta razón, se emplea una nueva columna "ambientes" que contabiliza la cantidad de ambientes en función de los dormitorios y lo transforma en un numero entero.

In [4]:
df['dormitorios'] = df['dormitorios'].fillna(0)
df['ambientes'] = df['dormitorios'].astype(str).str.extract('(\d+)').astype(int) + 1
df['ambientes'] 

0       2
1       1
2       1
3       1
4       2
       ..
1975    2
1976    3
1977    2
1978    2
1979    4
Name: ambientes, Length: 1923, dtype: int32

Vemos si hay dptos sin infor de superficie cubierta en m^2. En ese caso, rellenamos este dato con la mediana de los departamentos con esa cantidad de ambientes. El resto será descartado.

In [5]:

# convert area_m2 to numeric (extract numbers, handle commas and coercion)
df['area_m2'] = (
    df['area_m2']
    .astype(str)
    .str.extract(r'(\d+(?:[\.,]\d+)?)', expand=False)
    .str.replace(',', '.', regex=False)
)
df['area_m2'] = pd.to_numeric(df['area_m2'], errors='coerce')

mask = df['area_m2'].isna() & df['ambientes'].notna()

df.loc[mask, 'area_m2'] = (
    df.groupby('ambientes')['area_m2']
      .transform('median')
      .loc[mask]
)



In [6]:
df['area_m2'].isna().sum()

0

In [7]:

m = folium.Map(
    location=[df["lat"].mean(), df["lon"].mean()],
    zoom_start=12,
    tiles="CartoDB positron"
)

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=4,
        popup=f"${row['precio']}",
        color="blue",
        fill=True,
        fill_opacity=0.6
    ).add_to(m)

m

# Formulamos y entrenamos modelo

En cada punto espacial, vamos a modelar al precio por metro cuadrádo del alquiler como: p/M2 = \hat_K p/M2 + \beta_{Ambientes}(ambientes) + \beta_{m2}(m2) 


In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split

In [ ]:
import numpy as np
import statsmodels.formula.api as smf
from sklearn.neighbors import NearestNeighbors


class SpatialKernelPriceModel:
    def __init__(self, k_neighbors=30):
        self.k = k_neighbors
        self.model_ = None
        self.coords_ = None
        self.y_ = None
        self.nbrs_ = None
        self.h_ = None

    def _compute_kernel(self, coords_query):
        distances, indices = self.nbrs_.kneighbors(np.radians(coords_query))
        distances = distances[:, 1:]
        indices = indices[:, 1:]

        weights = np.exp(-(distances**2) / (2 * self.h_**2))
        weights /= weights.sum(axis=1, keepdims=True)

        return np.sum(weights * self.y_[indices], axis=1)

    def fit(self, df):
        self.coords_ = df[['lat', 'lon']].values
        self.y_ = df['precio_m2'].values

        self.nbrs_ = NearestNeighbors(
            n_neighbors=self.k + 1,
            metric='haversine'
        ).fit(np.radians(self.coords_))

        distances, _ = self.nbrs_.kneighbors(np.radians(self.coords_))
        self.h_ = np.median(distances[:, 1:])

        df = df.copy()
        df['K_hat'] = self._compute_kernel(self.coords_)

        self.model_ = smf.ols(
            "precio_m2 ~ K_hat + C(ambientes) + area_m2",
            data=df
        ).fit()

        return self

    def predict(self, df_new):
        if self.model_ is None:
            raise RuntimeError("No existe el modelo aún. Ajusta el modelo primero con fit().")

        coords_new = df_new[['lat', 'lon']].values
        df_new = df_new.copy()
        df_new['K_hat'] = self._compute_kernel(coords_new)

        return self.model_.predict(df_new)
